# LeetCode 1792 — Maximum Average Pass Ratio bằng Python

Link đề: https://leetcode.com/problems/maximum-average-pass-ratio/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def maxAverageRatio(self, classes: List[List[int]], extraStudents: int) -> float:
        ...
```

## 1. Mục tiêu học

1. Biến "tối đa trung bình" thành "tối đa **tổng** các tỉ lệ" và tính **lợi ích biên** (marginal gain) khi thêm một học sinh vào một lớp.
2. Chứng minh lợi ích biên của mỗi lớp **giảm dần** ⇒ tham lam "mỗi học sinh vào lớp có lợi ích biên lớn nhất" là tối ưu.
3. Cài tham lam bằng **heap lớn nhất** (`heapq` với khóa âm) — `O((n + k) log n)`.
4. So sánh với vét cạn mọi cách phân bổ (input nhỏ) và với tham lam quét tuyến tính `O(n·k)`.
5. Kiểm thử đáp án **số thực** bằng sai số tuyệt đối `1e-5`.

## 2. Đề bài & ví dụ

Có các lớp `classes[i] = [pass_i, total_i]`: lớp `i` có `total_i` học sinh, trong đó `pass_i` em sẽ đậu. Có thêm `extraStudents` học sinh giỏi — **chắc chắn đậu** ở bất kỳ lớp nào được xếp vào. Xếp mỗi học sinh giỏi vào một lớp để **trung bình tỉ lệ đậu** của các lớp là lớn nhất.

Tỉ lệ đậu của lớp = số đậu / tổng số. Trung bình = tổng các tỉ lệ / số lớp. Trả về trung bình lớn nhất; chấp nhận sai số `10⁻⁵`.

**Ví dụ 1:** `classes = [[1,2],[3,5],[2,2]], extraStudents = 2` → `0.78333`: xếp cả 2 em vào lớp đầu: `(3/4 + 3/5 + 2/2)/3 = 0.78333`.

**Ví dụ 2:** `classes = [[2,4],[3,9],[4,5],[2,10]], extraStudents = 4` → `0.53485`.

**Ràng buộc:** `1 ≤ classes.length ≤ 10⁵`; `1 ≤ pass_i ≤ total_i ≤ 10⁵`; `1 ≤ extraStudents ≤ 10⁵`.

In [1]:
import random
import time
import heapq
from fractions import Fraction
from typing import List

EPS = 1e-5
examples = [([[1, 2], [3, 5], [2, 2]], 2, 0.78333), ([[2, 4], [3, 9], [4, 5], [2, 10]], 4, 0.53485)]
def close(x, y, eps=EPS):
    return abs(x - y) <= eps
for c, k, exp in examples:
    print(f"classes={c}, extra={k} -> kỳ vọng ≈ {exp}")

classes=[[1, 2], [3, 5], [2, 2]], extra=2 -> kỳ vọng ≈ 0.78333
classes=[[2, 4], [3, 9], [4, 5], [2, 10]], extra=4 -> kỳ vọng ≈ 0.53485


## 3. Kiến thức nền

### 3.1 Lợi ích biên khi thêm một học sinh đậu

Lớp `(a, b)` (đậu `a` / tổng `b`) thêm một học sinh đậu thành `(a+1, b+1)`:

`Δ(a, b) = (a+1)/(b+1) − a/b = (b − a) / (b·(b+1))`.

Lớp đã đậu 100% (`a = b`) có `Δ = 0` — không nên thêm.

In [2]:
def gain(a, b):
    return (a + 1) / (b + 1) - a / b

for a, b in [(1, 2), (3, 5), (2, 2), (1, 10)]:
    print((a, b), f"Δ = {gain(a, b):.5f} = (b-a)/(b(b+1)) = {(b - a) / (b * (b + 1)):.5f}")

(1, 2) Δ = 0.16667 = (b-a)/(b(b+1)) = 0.16667
(3, 5) Δ = 0.06667 = (b-a)/(b(b+1)) = 0.06667
(2, 2) Δ = 0.00000 = (b-a)/(b(b+1)) = 0.00000
(1, 10) Δ = 0.08182 = (b-a)/(b(b+1)) = 0.08182


### 3.2 Heap lớn nhất bằng `heapq`

`heapq` là min-heap ⇒ đẩy `(−Δ, a, b)` để lấy lớp có `Δ` lớn nhất. `heapq.heapreplace(h, x)` = pop rồi push trong một thao tác.

In [3]:
h = [(-gain(a, b), a, b) for a, b in [[1, 2], [3, 5], [2, 2]]]
heapq.heapify(h)
print("lớp có lợi ích lớn nhất:", h[0][1:], "Δ =", -h[0][0])

lớp có lợi ích lớn nhất: (1, 2) Δ = 0.16666666666666663


### 3.3 So sánh số thực với sai số

Đáp án là `float`; ví dụ in làm tròn 5 chữ số. Kiểm tra bằng `abs(x − y) ≤ 1e-5`, không so `==`.

### 3.4 `Fraction` — tính chính xác tuyệt đối

`fractions.Fraction` cho phép tính tỉ lệ **chính xác** (không sai số) — dùng trong vét cạn để làm "đáp án chuẩn".

In [4]:
print(Fraction(3, 4) + Fraction(3, 5) + Fraction(2, 2), float((Fraction(3, 4) + Fraction(3, 5) + 1) / 3))

47/20 0.7833333333333333


## 4. Ý tưởng nền tảng / trực giác

Số lớp `n` cố định ⇒ tối đa trung bình ⇔ tối đa **tổng** tỉ lệ.

Tính tay **Ví dụ 1** (`extra = 2`):

| lớp | tỉ lệ | Δ khi thêm 1 | Δ khi thêm em thứ 2 |
|---|---|---|---|
| (1,2) | 0.5 | 2/3 − 1/2 = 0.1667 | 3/4 − 2/3 = 0.0833 |
| (3,5) | 0.6 | 4/6 − 3/5 = 0.0667 | 5/7 − 4/6 = 0.0476 |
| (2,2) | 1.0 | 0 | 0 |

- Em thứ nhất: Δ lớn nhất là lớp (1,2) ⇒ thành (2,3).
- Em thứ hai: so (2,3) Δ = 0.0833 với (3,5) Δ = 0.0667 ⇒ lại vào lớp đầu ⇒ (3,4).

Tổng `3/4 + 3/5 + 1 = 2.35` ⇒ trung bình `0.78333`.

**Quan sát:** Δ của cùng một lớp **giảm** sau mỗi lần thêm (0.1667 → 0.0833) — "lợi ích giảm dần".

In [5]:
a, b = 1, 2
print("Δ liên tiếp của lớp (1,2):", [round(gain(a + t, b + t), 5) for t in range(6)])

Δ liên tiếp của lớp (1,2): [0.16667, 0.08333, 0.05, 0.03333, 0.02381, 0.01786]


## 5. Cách 1 — Brute Force / baseline: thử mọi cách phân bổ

**Ý tưởng:** liệt kê mọi bộ `(x_1, …, x_n)` với `x_i ≥ 0`, `Σ x_i = k`; tính tổng `Σ (a_i + x_i)/(b_i + x_i)` bằng `Fraction` (chính xác), lấy max.

In [6]:
def compositions(k, n):
    if n == 1:
        yield (k,); return
    for x in range(k + 1):
        for rest in compositions(k - x, n - 1):
            yield (x,) + rest

def ratio_bruteforce(classes, extra):
    best = None
    for xs in compositions(extra, len(classes)):
        s = sum(Fraction(a + x, b + x) for (a, b), x in zip(classes, xs))
        if best is None or s > best:
            best = s
    return float(best / len(classes))

for c, k, exp in examples:
    got = ratio_bruteforce(c, k); print(f"{got:.5f} kỳ vọng {exp}"); assert close(got, exp)

0.78333 kỳ vọng 0.78333
0.53485 kỳ vọng 0.53485


**Độ phức tạp:** số cách phân bổ là `C(k + n − 1, n − 1)` — bùng nổ tổ hợp. Chỉ dùng cho `n, k` nhỏ.

**Nút thắt:** ta không tận dụng cấu trúc "lợi ích giảm dần" — mà với cấu trúc đó, quyết định từng em một là đủ.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (lõm / lợi ích giảm dần):** với lớp cố định, `Δ(a + t, b + t)` **giảm** theo `t` (chứng minh ở mục 10). Hàm "tỉ lệ theo số học sinh thêm vào" là **lõm**.

**Insight 2 (tham lam):** với tổng các hàm lõm và một "ngân sách" `k` đơn vị, phân bổ từng đơn vị vào nơi có **lợi ích biên lớn nhất hiện tại** là tối ưu.

**Quy tắc:** lặp `k` lần: lấy lớp có `Δ` lớn nhất, thêm một học sinh, cập nhật `Δ` của lớp đó.

**Cấu trúc dữ liệu:** heap lớn nhất theo `Δ` ⇒ mỗi lần `O(log n)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam quét tuyến tính (không heap)

In [7]:
def ratio_greedy_scan(classes, extra):
    cls = [list(c) for c in classes]
    for _ in range(extra):
        i = max(range(len(cls)), key=lambda t: gain(*cls[t]))
        cls[i][0] += 1; cls[i][1] += 1
    return sum(a / b for a, b in cls) / len(cls)

print([round(ratio_greedy_scan(c, k), 5) for c, k, _ in examples])

[0.78333, 0.53485]


**Code nộp LeetCode (Cách 2):** đúng nhưng `O(n · k) = 10¹⁰` ⇒ TLE với giới hạn lớn; chỉ để hiểu ý tưởng.

### 7.2 Cách 3 — Tham lam + heap lớn nhất

In [8]:
def ratio_heap(classes, extra):
    h = [(-gain(a, b), a, b) for a, b in classes]
    heapq.heapify(h)
    for _ in range(extra):
        _, a, b = h[0]
        a += 1; b += 1
        heapq.heapreplace(h, (-gain(a, b), a, b))
    return sum(a / b for _, a, b in h) / len(h)

print([round(ratio_heap(c, k), 5) for c, k, _ in examples])

[0.78333, 0.53485]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** `O(n + k log n)`, Space `O(n)`.

### 7.3 Cách 4 — Heap với công thức Δ rút gọn (ổn định số học hơn)

`Δ = (b − a) / (b(b+1))` tính bằng **một** phép chia thay vì hiệu hai số gần nhau (giảm sai số triệt tiêu). Ngoài ra, các lớp đã đầy (`a = b`) có `Δ = 0` mãi mãi ⇒ có thể bỏ khỏi heap ngay từ đầu.

In [9]:
class SolutionStable:
    def maxAverageRatio(self, classes: List[List[int]], extraStudents: int) -> float:
        n = len(classes)
        full = sum(1 for a, b in classes if a == b)            # lớp 100%: đóng góp 1, không cần thêm
        h = [(-(b - a) / (b * (b + 1)), a, b) for a, b in classes if a < b]
        if not h:
            return 1.0
        heapq.heapify(h)
        for _ in range(extraStudents):
            _, a, b = h[0]
            a += 1; b += 1
            heapq.heapreplace(h, (-(b - a) / (b * (b + 1)), a, b))
        return (full + sum(a / b for _, a, b in h)) / n

print([round(SolutionStable().maxAverageRatio(c, k), 5) for c, k, _ in examples])

[0.78333, 0.53485]


**Độ phức tạp Cách 4:** như Cách 3; heap có thể nhỏ hơn (bỏ lớp đầy).

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 2** (`extra = 4`): mỗi bước in lớp được chọn và Δ.

In [10]:
def trace_heap(classes, extra):
    h = [(-gain(a, b), a, b, i) for i, (a, b) in enumerate(classes)]
    heapq.heapify(h)
    print("Δ ban đầu:", [(i, (a, b), round(-g, 5)) for g, a, b, i in sorted(h)])
    for step in range(1, extra + 1):
        g, a, b, i = h[0]
        print(f"bước {step}: chọn lớp {i} ({a},{b}) với Δ={-g:.5f} -> ({a+1},{b+1}), Δ mới={gain(a+1, b+1):.5f}")
        heapq.heapreplace(h, (-gain(a + 1, b + 1), a + 1, b + 1, i))
    res = sum(a / b for _, a, b, _ in h) / len(h)
    print("Trạng thái cuối:", sorted((i, (a, b)) for _, a, b, i in h), f"-> trung bình {res:.5f}")
    return res

assert close(trace_heap(*examples[1][:2]), examples[1][2])

Δ ban đầu: [(0, (2, 4), 0.1), (3, (2, 10), 0.07273), (1, (3, 9), 0.06667), (2, (4, 5), 0.03333)]
bước 1: chọn lớp 0 (2,4) với Δ=0.10000 -> (3,5), Δ mới=0.06667
bước 2: chọn lớp 3 (2,10) với Δ=0.07273 -> (3,11), Δ mới=0.06061
bước 3: chọn lớp 1 (3,9) với Δ=0.06667 -> (4,10), Δ mới=0.05455
bước 4: chọn lớp 0 (3,5) với Δ=0.06667 -> (4,6), Δ mới=0.04762
Trạng thái cuối: [(0, (4, 6)), (1, (4, 10)), (2, (4, 5)), (3, (3, 11))] -> trung bình 0.53485


| bước | lớp chọn (trước → sau) | Δ của lựa chọn | Δ mới của lớp đó |
|---|---|---|---|
| 1 | lớp 0: (2,4) → (3,5) | 0.10000 | 0.06667 |
| 2 | lớp 3: (2,10) → (3,11) | 0.07273 | 0.06061 |
| 3 | lớp 1: (3,9) → (4,10) | 0.06667 | 0.05455 |
| 4 | lớp 0: (3,5) → (4,6) | 0.06667 | 0.04762 |

Bước 3 và 4 hòa Δ = 0.06667 (`(3,9)` và `(3,5)` đều có `(b−a)/(b(b+1))` = 6/90 = 2/30); heap lấy một trong hai trước — thứ tự không ảnh hưởng tổng. Kết quả: `(4/6 + 4/10 + 4/5 + 3/11)/4 ≈ 0.53485`.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — tham lam + heap.

In [11]:
class Solution:
    def maxAverageRatio(self, classes: List[List[int]], extraStudents: int) -> float:
        def gain(a: int, b: int) -> float:
            return (b - a) / (b * (b + 1))
        h = [(-gain(a, b), a, b) for a, b in classes]
        heapq.heapify(h)
        for _ in range(extraStudents):
            _, a, b = h[0]
            heapq.heapreplace(h, (-gain(a + 1, b + 1), a + 1, b + 1))
        return sum(a / b for _, a, b in h) / len(classes)

for c, k, exp in examples:
    got = Solution().maxAverageRatio(c, k); print(f"{got:.5f} {exp}"); assert close(got, exp)

0.78333 0.78333
0.53485 0.53485


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (lợi ích giảm dần).** `Δ(a, b) = (b − a)/(b(b+1))`. Sau khi thêm: `Δ(a+1, b+1) = (b − a)/((b+1)(b+2))`. Vì `b(b+1) < (b+1)(b+2)` và tử số không đổi (`≥ 0`) ⇒ `Δ(a+1, b+1) ≤ Δ(a, b)`. Vậy với mỗi lớp, dãy lợi ích biên `Δ_1 ≥ Δ_2 ≥ …` không tăng.

**Bổ đề 2 (tham lam tối ưu cho tổng hàm lõm).** Tổng tỉ lệ khi lớp `i` nhận `x_i` học sinh = `r_i + Σ_{t=1}^{x_i} Δ_{i,t}`. Bài toán thành: chọn **đúng `k`** phần tử từ các dãy giảm `Δ_{i,1} ≥ Δ_{i,2} ≥ …`, trong mỗi dãy phải lấy một **tiền tố**, để tổng lớn nhất. Vì mỗi dãy giảm, chọn `k` giá trị lớn nhất trong **hợp** mọi dãy luôn tự động là các tiền tố (nếu lấy `Δ_{i,t}` thì mọi `Δ_{i,s}`, `s < t`, lớn hơn hoặc bằng cũng được lấy — với hòa có thể sắp lại). Tham lam bằng heap chọn đúng `k` giá trị lớn nhất đó theo thứ tự. ∎

**Heap đúng:** đỉnh heap luôn chứa lợi ích biên **hiện tại** lớn nhất trong các lớp, vì mỗi lần ta chỉ thay đổi lớp vừa được chọn và cập nhật đúng khóa của nó.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(classes) ≤ 10⁵`, `k = extraStudents ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi phân bổ | `O(C(k+n−1, n−1) · n)` | `O(n)` |
| 2. Tham lam quét | `O(n · k)` | `O(n)` |
| 3. Tham lam + heap | `O(n + k log n)` | `O(n)` |
| 4. Heap ổn định số học | `O(n + k log n)` | `O(n)` |

In [12]:
rng = random.Random(1792)
big = [[a, rng.randint(a, 10 ** 5)] for a in (rng.randint(1, 10 ** 5) for _ in range(10 ** 5))]
for name, f in [("Cách 3", Solution().maxAverageRatio), ("Cách 4", SolutionStable().maxAverageRatio)]:
    t = time.perf_counter(); r = f(big, 10 ** 5); print(f"{name}: {r:.6f} ({(time.perf_counter()-t)*1000:.0f} ms)")
small = big[:2000]
t = time.perf_counter(); r = ratio_greedy_scan(small, 300); print(f"Cách 2 (n=2000, k=300): {r:.6f} ({(time.perf_counter()-t)*1000:.0f} ms) — n=k=10^5 sẽ ~10^10 phép")

Cách 3: 0.643921 (114 ms)


Cách 4: 0.643921 (82 ms)
Cách 2 (n=2000, k=300): 0.645879 (62 ms) — n=k=10^5 sẽ ~10^10 phép


## 12. Bộ test

Đáp án là số thực ⇒ so bằng `close(·, ·, 1e-5)`. Đối chiếu với vét cạn chính xác (`Fraction`) trên input nhỏ; đối chiếu giữa các cách nhanh trên input lớn hơn.

In [13]:
ALL = {"scan": ratio_greedy_scan, "heap": ratio_heap, "stable": SolutionStable().maxAverageRatio, "submit": Solution().maxAverageRatio}
def run_all(classes, extra, with_brute=False):
    res = {k: f([list(c) for c in classes], extra) for k, f in ALL.items()}
    ref = ratio_bruteforce(classes, extra) if with_brute else res["heap"]
    for k, v in res.items():
        assert close(v, ref, 1e-9), (k, classes, extra, res, ref)
    return ref

for c, k, exp in examples:
    assert close(run_all(c, k, with_brute=True), exp)
edge = [([[1, 1]], 1, 1.0), ([[1, 2]], 1, 2 / 3), ([[1, 2]], 3, 0.8), ([[2, 2], [1, 3]], 1, (1 + 0.5) / 2), ([[1, 100000]], 1, 2 / 100001)]
for c, k, exp in edge:
    assert close(run_all(c, k, with_brute=True), exp), (c, k)
rng = random.Random(17920)
for _ in range(250):
    n = rng.randint(1, 4); k = rng.randint(1, 6)
    cls = [[a, rng.randint(a, 12)] for a in (rng.randint(1, 12) for _ in range(n))]
    run_all(cls, k, with_brute=True)
for _ in range(100):
    n = rng.randint(1, 60); k = rng.randint(1, 200)
    run_all([[a, rng.randint(a, 1000)] for a in (rng.randint(1, 1000) for _ in range(n))], k)
print("Ví dụ, edge case, 250 test vét cạn chính xác, 100 test lớn: OK")

Ví dụ, edge case, 250 test vét cạn chính xác, 100 test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam theo **tỉ lệ thấp nhất** (thay vì lợi ích biên)

In [14]:
def wrong_lowest_ratio(classes, extra):
    h = [(a / b, a, b) for a, b in classes]; heapq.heapify(h)
    for _ in range(extra):
        _, a, b = h[0]; heapq.heapreplace(h, ((a + 1) / (b + 1), a + 1, b + 1))
    return sum(a / b for _, a, b in h) / len(h)

c, k = [[1, 1000], [1, 3]], 1                    # ví dụ tự dựng (đối chiếu lại đề gốc)
print(f"sai : {wrong_lowest_ratio(c, k):.5f} | đúng: {Solution().maxAverageRatio(c, k):.5f} | brute: {ratio_bruteforce(c, k):.5f}")

sai : 0.16767 | đúng: 0.25050 | brute: 0.25050


**Vì sao sai:** lớp tỉ lệ thấp nhất `(1, 1000)` có **lợi ích biên rất nhỏ** (`999/(1000·1001) ≈ 0.001`) vì quá đông; lớp `(1, 3)` tăng `0.25 − …` = `1/6`. **Sửa:** chọn theo `Δ`.

### Lỗi 2 — Tham lam theo số học sinh **trượt** nhiều nhất (`b − a`)

In [15]:
def wrong_most_fail(classes, extra):
    cls = [list(c) for c in classes]
    for _ in range(extra):
        i = max(range(len(cls)), key=lambda t: cls[t][1] - cls[t][0])
        cls[i][0] += 1; cls[i][1] += 1
    return sum(a / b for a, b in cls) / len(cls)

c, k = [[1, 1000], [1, 3]], 1
print(f"sai : {wrong_most_fail(c, k):.5f} | đúng: {Solution().maxAverageRatio(c, k):.5f}")

sai : 0.16767 | đúng: 0.25050


**Vì sao sai:** `b − a` không đổi khi thêm học sinh đậu, nên mọi em đều dồn vào cùng một lớp; và tử số `b − a` lớn chưa chắc `Δ` lớn (mẫu `b(b+1)` cũng lớn). **Sửa:** dùng đúng `Δ = (b − a)/(b(b+1))`.

### Lỗi 3 — Không cập nhật Δ sau khi thêm (tính Δ một lần rồi chia đều theo thứ tự)

In [16]:
def wrong_static(classes, extra):
    order = sorted(range(len(classes)), key=lambda i: -gain(*classes[i]))
    cls = [list(c) for c in classes]
    for t in range(extra):
        i = order[0]                               # SAI: luôn dồn vào lớp có Δ ban đầu lớn nhất
        cls[i][0] += 1; cls[i][1] += 1
    return sum(a / b for a, b in cls) / len(cls)

c, k, exp = examples[1]
print(f"sai : {wrong_static(c, k):.5f} | đúng: {exp}")

sai : 0.52083 | đúng: 0.53485


**Vì sao sai:** Δ của lớp vừa được thêm **giảm**; lớp khác có thể trở nên tốt hơn. **Sửa:** cập nhật Δ và đưa lớp trở lại heap sau mỗi lần thêm.

### Lỗi 4 — Chia nguyên `a // b` khi tính tỉ lệ

In [17]:
def wrong_int_div(classes, extra):
    h = [(-gain(a, b), a, b) for a, b in classes]; heapq.heapify(h)
    for _ in range(extra):
        _, a, b = h[0]; heapq.heapreplace(h, (-gain(a + 1, b + 1), a + 1, b + 1))
    return sum(a // b for _, a, b in h) / len(h)          # SAI: // cho 0 hoặc 1

c, k, exp = examples[0]
print("sai :", wrong_int_div(c, k), "| đúng:", round(Solution().maxAverageRatio(c, k), 5))

sai : 0.3333333333333333 | đúng: 0.78333


**Vì sao sai:** `3 // 4 = 0`. **Sửa:** chia số thực `/`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phân bổ `k` đơn vị tài nguyên để tối đa tổng các hàm" + mỗi hàm có **lợi ích biên giảm dần** ⇒ **tham lam + heap** theo lợi ích biên.
- Lợi ích biên phải **cập nhật** sau mỗi lần cấp phát.
- Tối đa trung bình với số nhóm cố định ⇔ tối đa tổng.
- Đáp án số thực ⇒ test bằng sai số; vét cạn dùng `Fraction`.

## 15. Biến thể của bài

1. **Mỗi học sinh giỏi có xác suất đậu `p`**: lợi ích biên `(a+p)/(b+1) − a/b` vẫn giảm dần ⇒ cùng tham lam.
2. **Mỗi lớp nhận tối đa `c_i` học sinh**: bỏ lớp khỏi heap khi đạt giới hạn.
3. **Tối đa tỉ lệ nhỏ nhất** (maximin): khác hẳn — tham lam vào lớp có tỉ lệ thấp nhất, hoặc nhị phân trên đáp án.
4. **1642 Furthest Building You Can Reach**, **2233 Maximum Product After K Increments**, **1834**: tham lam + heap theo lợi ích.
5. **Trả về cách phân bổ** (Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi phân bổ | Vét cạn chính xác | tổ hợp | `O(n)` | Kiểm chứng |
| 2. Tham lam quét | Δ lớn nhất, quét mảng | `O(nk)` | `O(n)` | `n, k` nhỏ |
| 3. Tham lam + heap | Δ lớn nhất qua heap | `O(n + k log n)` | `O(n)` | **Bản nộp** |
| 4. Heap ổn định | Δ rút gọn, bỏ lớp đầy | `O(n + k log n)` | `O(n)` | Khi lo sai số |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([[1,1]], 5)`; (b) `([[1,2]], 2)`; (c) `([[1,2],[1,2]], 2)`; (d) `([[1,3],[2,3]], 1)` — làm tròn 5 chữ số.

**Bài 2 (tính tay):** Với `classes = [[1,3],[1,4]]`, `extra = 2`: tính Δ ban đầu, chọn từng bước, kết quả.

**Bài 3 (điền TODO):** Hoàn thiện `gain_todo(a, b)` dạng rút gọn `(b − a)/(b(b+1))`.

**Bài 4 (tự viết lại):** Viết `allocation(classes, extra)` trả list `x_i` số học sinh giỏi xếp vào mỗi lớp theo tham lam; kiểm tra `Σ x_i = extra` và tổng tỉ lệ đúng.

**Bài 5 (mini challenge):** Mỗi lớp có thêm giới hạn `cap[i]` học sinh giỏi tối đa; nếu mọi lớp đã đầy giới hạn thì số học sinh còn lại bỏ. Viết `max_avg_cap(classes, extra, cap)`, đối chiếu vét cạn.

In [18]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def gain_todo(a, b):
    # TODO: trả (b - a) / (b * (b + 1))
    pass

In [19]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def allocation(classes, extra):
    # TODO: heap (-Δ, chỉ số, a, b); mỗi lần chọn thì x[chỉ số] += 1
    pass

def max_avg_cap(classes, extra, cap):
    # TODO: như heap nhưng khi lớp đạt cap thì không đẩy lại vào heap
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: Δ. Mức 2: tham lam. Mức 3: (a) `1.0`; (b) `3/4 = 0.75`; (c) mỗi lớp một em: `2/3 ≈ 0.66667`; (d) lớp `(1,3)` Δ = 1/6 > lớp `(2,3)` Δ = 1/12 ⇒ `(2/4 + 2/3)/2 ≈ 0.58333`.

**Bài 2.** Mức 1: Δ(1,3) = 2/12 = 0.1667, Δ(1,4) = 3/20 = 0.15. Mức 2: em 1 vào (1,3) → (2,4), Δ mới = 2/20 = 0.1 < 0.15 ⇒ em 2 vào (1,4) → (2,5). Mức 3: `(2/4 + 2/5)/2 = 0.45`.

**Bài 3.** Mức 1: hiệu tử. Mức 2: tích mẫu. Mức 3: so với `gain(a, b)` trong sai số.

**Bài 4.** Mức 1: thêm chỉ số vào tuple heap. Mức 2: `x[i] += 1`. Mức 3: tổng `Σ (a_i + x_i)/(b_i + x_i)` khớp `Solution` × `n`.

**Bài 5.** Mức 1: đếm `used[i]`. Mức 2: khi `used[i] == cap[i]` thì `heappop` thay vì `heapreplace`. Mức 3: heap rỗng ⇒ dừng. Vét cạn: `x_i ≤ cap_i`, `Σ x_i ≤ extra` (có thể không dùng hết khi bị giới hạn) — nhưng vì Δ ≥ 0, dùng càng nhiều càng tốt.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
b1 = [round(Solution().maxAverageRatio(c, k), 5) for c, k in [([[1, 1]], 5), ([[1, 2]], 2), ([[1, 2], [1, 2]], 2), ([[1, 3], [2, 3]], 1)]]
print("Bài 1:", b1); assert b1 == [1.0, 0.75, 0.66667, 0.58333]
r2 = Solution().maxAverageRatio([[1, 3], [1, 4]], 2); print("Bài 2:", r2); assert close(r2, 0.45)
def gain_todo(a, b):
    return (b - a) / (b * (b + 1))
assert all(close(gain_todo(a, b), gain(a, b), 1e-12) for b in range(1, 60) for a in range(1, b + 1))
print("Bài 3: OK")

Bài 1: [1.0, 0.75, 0.66667, 0.58333]
Bài 2: 0.45
Bài 3: OK


In [21]:
# Bài 4, 5
def allocation(classes, extra):
    x = [0] * len(classes)
    h = [(-gain(a, b), i, a, b) for i, (a, b) in enumerate(classes)]; heapq.heapify(h)
    for _ in range(extra):
        _, i, a, b = h[0]; x[i] += 1
        heapq.heapreplace(h, (-gain(a + 1, b + 1), i, a + 1, b + 1))
    return x

def max_avg_cap(classes, extra, cap):
    used = [0] * len(classes); cls = [list(c) for c in classes]
    h = [(-gain(a, b), i) for i, (a, b) in enumerate(classes) if cap[i] > 0]; heapq.heapify(h)
    for _ in range(extra):
        if not h: break
        _, i = h[0]
        cls[i][0] += 1; cls[i][1] += 1; used[i] += 1
        if used[i] == cap[i]: heapq.heappop(h)
        else: heapq.heapreplace(h, (-gain(*cls[i]), i))
    return sum(a / b for a, b in cls) / len(cls)

def cap_brute(classes, extra, cap):
    best = None
    def rec(i, left, acc):
        nonlocal best
        if i == len(classes):
            best = acc if best is None or acc > best else best; return
        a, b = classes[i]
        for x in range(min(cap[i], left) + 1):
            rec(i + 1, left - x, acc + Fraction(a + x, b + x))
    rec(0, extra, Fraction(0))
    return float(best / len(classes))

rng = random.Random(7)
for _ in range(200):
    cls = [[a, rng.randint(a, 9)] for a in (rng.randint(1, 9) for _ in range(rng.randint(1, 4)))]
    k = rng.randint(1, 6)
    x = allocation(cls, k)
    assert sum(x) == k and close(sum((a + t) / (b + t) for (a, b), t in zip(cls, x)) / len(cls), Solution().maxAverageRatio(cls, k), 1e-9)
    cap = [rng.randint(0, 3) for _ in cls]
    assert close(max_avg_cap(cls, k, cap), cap_brute(cls, k, cap), 1e-9), (cls, k, cap)
print("Bài 4, 5: OK |", allocation(*examples[1][:2]), round(max_avg_cap([[1, 2], [3, 5], [2, 2]], 2, [1, 1, 1]), 5))

Bài 4, 5: OK | [2, 1, 0, 1] 0.77778


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tham lam theo lợi ích biên + heap":**

```python
gain = lambda a, b: (b - a) / (b * (b + 1))
h = [(-gain(a, b), a, b) for a, b in classes]; heapify(h)
for _ in range(extra):
    _, a, b = h[0]
    heapreplace(h, (-gain(a + 1, b + 1), a + 1, b + 1))
return sum(a / b for _, a, b in h) / len(classes)
```

**Checklist:**
- [ ] Chọn theo **lợi ích biên** Δ, không theo tỉ lệ thấp nhất hay số trượt?
- [ ] **Cập nhật** Δ sau mỗi lần thêm (đẩy lại vào heap)?
- [ ] Heap lớn nhất (khóa âm)?
- [ ] Chia số thực `/`?
- [ ] Đã test lớp 100%, một lớp, `extra` lớn, so sánh với sai số `1e-5`?